In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

image_features_path = PROCESSED_DIR / "image_features.csv"
excel_data_path = PROCESSED_DIR / "excel_features_clean.csv"

In [3]:
image_features = pd.read_csv(image_features_path)
excel_data = pd.read_csv(excel_data_path)

print("Image features:", image_features.shape)
print("Excel data:", excel_data.shape)

image_features.head()

Image features: (89, 31)
Excel data: (89, 34)


,case_id,image_width,image_height,is_landscape,ink_pixels,ink_density,stroke_darkness_mean,stroke_darkness_std,skeleton_length,skeleton_density,...,components_per_ink_area,bbox_width,bbox_height,bbox_area_ratio,bbox_aspect_ratio,bbox_center_x_norm,bbox_center_y_norm,drawing_center_distance_norm,perseveration_score,distortion_proxy_score
0,1,1241,1755,0,27946,0.012831,63.727796,49.882035,7015,0.003221,...,0.004723,1200,1687,0.929496,0.711322,0.493554,0.511681,0.010238,-1.366521,-0.303671
1,2,1241,1755,0,33761,0.015501,117.697965,63.608772,6718,0.003085,...,0.004206,1183,1643,0.892428,0.720024,0.500403,0.478632,0.017448,0.074930,0.471576
2,3,1241,1755,0,29999,0.013774,81.395313,53.612199,7692,0.003532,...,0.004067,1165,1494,0.799149,0.779786,0.478646,0.433903,0.055358,-1.116456,-0.379011
3,4,1755,1241,1,31808,0.014605,94.960670,47.822996,7157,0.003286,...,0.004464,1512,1155,0.801835,1.309091,0.511396,0.489122,0.011226,-0.597039,-0.231889
4,5,1241,1755,0,33813,0.015525,125.515246,64.123418,7413,0.003404,...,0.004081,1191,1571,0.859091,0.758116,0.497180,0.479202,0.017059,-0.059099,-0.291300


In [4]:
X_img = image_features.drop(columns=["case_id"])
y = excel_data["target"]

print("X image shape:", X_img.shape)
print("Target distribution:")
print(y.value_counts())

X image shape: (89, 30)
Target distribution:
target
سوي              84
العصاب            3
التخلف العقلي     2
Name: count, dtype: int64


In [5]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Classes:", label_encoder.classes_)

Classes: ['التخلف العقلي' 'العصاب' 'سوي']


In [18]:
cv = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

In [11]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=42
        ))
    ])
}

In [12]:
results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_img,
        y_encoded,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        return_train_score=False
    )
    
    results.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std()
    })

image_results_df = pd.DataFrame(results)
image_results_df

d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


,model,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std
0,Logistic Regression,0.798039,0.043127,0.376625,0.064820
1,Random Forest,0.943791,0.001307,0.485541,0.000346
2,SVM,0.785621,0.109924,0.405313,0.176917


In [13]:
best_image_model = models["Random Forest"]

y_pred_img = cross_val_predict(
    best_image_model,
    X_img,
    y_encoded,
    cv=cv
)

print(classification_report(
    y_encoded,
    y_pred_img,
    target_names=label_encoder.classes_
))

cm_img = confusion_matrix(y_encoded, y_pred_img)

cm_img_df = pd.DataFrame(
    cm_img,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

cm_img_df

d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


               precision    recall  f1-score   support

التخلف العقلي       0.00      0.00      0.00         2
       العصاب       0.00      0.00      0.00         3
          سوي       0.94      1.00      0.97        84

     accuracy                           0.94        89
    macro avg       0.31      0.33      0.32        89
 weighted avg       0.89      0.94      0.92        89



d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\FCAI\Year 3\2nd semster\Cognitive\bender-gestalt-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

,التخلف العقلي,العصاب,سوي
التخلف العقلي,0,0,2
العصاب,0,0,3
سوي,0,0,84


In [14]:
image_results_df.to_csv(
    PROCESSED_DIR / "image_features_ml_results.csv",
    index=False,
    encoding="utf-8-sig"
)

cm_img_df.to_csv(
    PROCESSED_DIR / "image_features_confusion_matrix.csv",
    encoding="utf-8-sig"
)

print("Saved image ML results.")

Saved image ML results.
